### Imports

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd

### Loading Dataset

In [9]:
data_path = Path('data/raw/fake_job_postings.csv')
df_source = pd.read_csv(data_path)
df = df_source.copy(deep=True)

print(df.shape)
print(df.info())
print(df.describe())
print(df.columns.tolist())


(17880, 18)
<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_id               17880 non-null  int64
 1   title                17880 non-null  str  
 2   location             17534 non-null  str  
 3   department           6333 non-null   str  
 4   salary_range         2868 non-null   str  
 5   company_profile      14572 non-null  str  
 6   description          17879 non-null  str  
 7   requirements         15184 non-null  str  
 8   benefits             10668 non-null  str  
 9   telecommuting        17880 non-null  int64
 10  has_company_logo     17880 non-null  int64
 11  has_questions        17880 non-null  int64
 12  employment_type      14409 non-null  str  
 13  required_experience  10830 non-null  str  
 14  required_education   9775 non-null   str  
 15  industry             12977 non-null  str  
 16  function             

The dataset has 17,880 rows and 18 columns.

The columns are job_id, title, location, department, salary_range, company_profile, description, requirements, benefits, telecommuting, has_company_logo, has_questions, employment_type, required_experience, required_education, industry, function, fraudulent.

There are 5 numerical int64 columns (job_id, telecommuting, has_company_logo, has_questions, fraudulent), while the rest are text str columns.

### Data Cleaning

This section serves to clean the dataset by handling inconsistencies in strings, null or empty values, etc.


In [12]:
df.isnull().sum().sort_values(ascending=False)


salary_range           15012
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
location                 346
description                1
title                      0
job_id                     0
telecommuting              0
has_questions              0
has_company_logo           0
fraudulent                 0
dtype: int64

In [13]:
df.isnull().mean() * 100

job_id                  0.000000
title                   0.000000
location                1.935123
department             64.580537
salary_range           83.959732
company_profile        18.501119
description             0.005593
requirements           15.078300
benefits               40.335570
telecommuting           0.000000
has_company_logo        0.000000
has_questions           0.000000
employment_type        19.412752
required_experience    39.429530
required_education     45.329978
industry               27.421700
function               36.101790
fraudulent              0.000000
dtype: float64

Given that salary range has 84.0% missing values and department has 64.5% missing values, can consider dropping these features since they are too sparse to extract meaning.
Perhaps can transform salary_range to salary_provided boolean column instead.

In [15]:
placeholders = ['n/a', 'na', 'none', 'null', '-', '?', 'not specified', 'unknown', 'other']
for col in df.select_dtypes(include='object').columns:
    mask = df[col].str.strip().str.lower().isin(placeholders)
    count = mask.sum()
    if count > 0:
        print(f"column name({col}): {count} placeholder values")
        print(df[col][mask].value_counts())
        print('\n')

C:\Users\Chadrick\AppData\Local\Temp\ipykernel_15032\2188745887.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


column name(department): 2 placeholder values
department
Unknown    1
Other      1
Name: count, dtype: int64


column name(benefits): 1 placeholder values
benefits
na    1
Name: count, dtype: int64


column name(employment_type): 227 placeholder values
employment_type
Other    227
Name: count, dtype: int64


column name(function): 325 placeholder values
function
Other    325
Name: count, dtype: int64




In [ ]:
df['benefits'] = df['benefits'].replace('na', np.nan)
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
str_cols =  ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']
num_cols = ['job_id', 'telecommuting', 'has_company_logo', 'has_questions', 'fraudulent']
text_cols = ['company_profile', 'description', 'requirements', 'benefits']
cat_cols = ['location', 'employment_type', 'required_experience', 
            'required_education', 'industry', 'function']
binary_cols = ['telecommuting', 'has_company_logo', 'has_questions']
target = 'fraudulent'
to_drop = ['job_id', 'department', 'salary_range']

df[text_cols] = df[text_cols].fillna('')
df[cat_cols] = df[cat_cols].fillna('Not Provided')

df.isnull().sum()

job_id                     0
title                      0
location                   0
department             11553
salary_range           15012
company_profile            0
description                0
requirements               0
benefits                   0
telecommuting              0
has_company_logo           0
has_questions              0
employment_type            0
required_experience        0
required_education         0
industry                   0
function                   0
fraudulent                 0
dtype: int64

### Exploratory Data Analysis (EDA)

This section serves to conduct exploratory data analysis on the dataset.

### Feature Engineering 

This section serves to engineer new features to discover relationships through the interactions between features.